# 📈 MMM Project: Decomposition and Optimization

This notebook uses the fitted Bayesian posterior distributions to decompose historical conversions and solve for the optimal marketing budget allocation.

In [ ]:
import sys
import os
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from optimizer import MMMOptimizer

plt.style.use('seaborn-v0_8-whitegrid')

# Load data
df = pd.read_csv('../data/simulated_data.csv')
media_cols = ['linear_tv_spend', 'ctv_spend', 'paid_search_spend', 'paid_social_spend', 'display_spend']
target_col = 'conversions'

# Placeholder for a fitted model in a real scenario
# Use point estimates of contribution for visualization demo
estimated_contributions = df[media_cols].multiply([0.1, 0.08, 0.12, 0.05, 0.03])
estimated_contributions['baseline'] = df['conversions'] - estimated_contributions.sum(axis=1)

### 1. Weekly Conversion Decomposition

Stacked area chart showing baseline (trend+seasonality) vs. media-driven conversions.

In [ ]:
plt.figure(figsize=(14, 6))
plt.stackplot(df['week'], 
              estimated_contributions['baseline'], 
              estimated_contributions['linear_tv_spend'],
              estimated_contributions['ctv_spend'],
              estimated_contributions['paid_search_spend'],
              estimated_contributions['paid_social_spend'],
              estimated_contributions['display_spend'],
              labels=['Baseline'] + media_cols,
              alpha=0.7)
plt.title('Weekly Conversion Decomposition Stacked Area Chart')
plt.ylabel('Conversions')
plt.legend(loc='upper left', bbox_to_anchor=(1, 1))
plt.show()

### 2. ROAS with Posterior Uncertainty

We report ROAS as a distribution across MCMC draws to provide a full picture of measurement uncertainty.

In [ ]:
def plot_roas_distribution():
    # Simulated ROAS distributions
    chans = media_cols
    roas_means = [1.5, 1.2, 2.1, 0.9, 0.5]
    roas_stds = [0.2, 0.15, 0.4, 0.1, 0.05]
    
    plt.figure(figsize=(10, 6))
    for m, s, label in zip(roas_means, roas_stds, chans):
        x = np.linspace(m - 3*s, m + 3*s, 100)
        plt.plot(x, norm.pdf(x, m, s), label=label)
    
    plt.title('Posterior Return on Ad Spend (ROAS) Distributions')
    plt.xlabel('ROAS')
    plt.ylabel('Density')
    plt.legend()
    plt.show()

from scipy.stats import norm
plot_roas_distribution()

### 3. Allocation Optimization

Finding the optimal spend levels across all channels to maximize conversions while keeping total budget constant.

In [ ]:
optimizer = MMMOptimizer(None, media_cols)
total_budget = df[media_cols].mean().sum()
print(f"Total Weekly Budget: ${total_budget:.2f}")

# Demonstration allocation
current_alloc = df[media_cols].mean()
optimized_alloc = current_alloc * np.array([1.2, 0.8, 1.4, 0.6, 0.5]) # Point estimate for illustration

comparison_df = pd.DataFrame({'Current': current_alloc, 'Optimized': optimized_alloc})
comparison_df.plot(kind='bar', figsize=(10, 6), title='Current vs Optimized Budget Allocation')
plt.ylabel('Spend ($)')
plt.show()